# Lab 01 — PyTorch Image Classification Inference
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chelsietao/ai-inference-lab/blob/main/notebooks/01_pytorch_inference.ipynb)

> **AI Inference Lab** · 用真實的 pretrained model 做 inference,並認識 inference 的執行流程。

## 這個 Lab 在做什麼?
載入 torchvision 的 **MobileNetV3-Small**(pretrained on ImageNet),對一張圖片做 **inference**,顯示 Top-5 預測與 confidence。你會親手跑過 inference 的完整流程:Device 偵測 → Model 載入 → Preprocessing → Forward pass → Softmax → Top-K。

## 學習目標 (Learning Objectives)
1. 了解什麼是 inference(只做 forward pass,不訓練)。
2. 能自動偵測 CPU / GPU 並讀懂 hardware 資訊。
3. 知道 preprocessing 為何必須與 model 的訓練設定一致。
4. 能分開看 preprocessing 與 model inference 的時間。

## 先備知識 (Prerequisites)
基本 Python 與機器學習概念即可,**不需要懂 CUDA 或 GPU 程式設計**。

## 需要 GPU 嗎?
**不需要。** 這個 Lab 在 CPU 上就能完整執行。有 GPU 會自動使用,沒有也不會出錯。

## 如何在 Colab 啟用 GPU(選用)
`Runtime` → `Change runtime type` → `Hardware accelerator` 選 **T4 GPU** → `Save`。Colab Free 不保證隨時有 GPU,沒有時請直接用 CPU。

## 如何執行
從上到下依序執行每個 cell(或 `Runtime` → `Run all`)。通常只需要修改第 1 節的參數。

## 預期會看到什麼
- 一份 hardware 摘要。
- 圖片與 Top-5 預測(label + confidence)。**實際數字會依你的環境而異,這裡不預先列出。**

## 如果沒有 GPU
程式會自動使用 CPU 並顯示說明。後面的 Lab 02 在沒有 GPU 時只會跑 CPU,不會產生任何假的 GPU 數據。

## 1. 參數 (Parameters)
只需要改這裡。`DEVICE` 可設 `"auto"` / `"cpu"` / `"cuda"`;`IMAGE_SOURCE` 設 `None` 會下載 PyTorch Hub 的範例圖片,也可填圖片路徑或 URL。

In [ ]:
DEVICE = "auto"        # "auto" | "cpu" | "cuda" | "mps"
IMAGE_SOURCE = None    # None = 範例圖片;或填入檔案路徑 / URL
TOP_K = 5

## 2. 安裝與載入套件
**做什麼**:把本專案的共用 Python package 載入 notebook。**為什麼**:所有 benchmark 程式碼都放在 `src/ai_inference_lab`,notebook 不重複貼 timing 程式碼。

In [ ]:
# 安裝 AI Inference Lab 套件(只在尚未安裝時執行;PyTorch / torchvision 在 Colab 已內建)
import importlib.util, subprocess, sys
if importlib.util.find_spec("ai_inference_lab") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/chelsietao/ai-inference-lab.git@main"])

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import ai_inference_lab as lab
print("ai_inference_lab", lab.__version__)

## 3. Hardware 偵測
**做什麼**:回報 device、CUDA 版本、PyTorch 版本、GPU 記憶體與支援的 precision。**為什麼**:不同硬體的結果不能直接比較,所以每筆結果都會記錄這些資訊。**常見錯誤**:假設 Colab 一定有 GPU。

In [ ]:
hw = lab.detect_hardware(DEVICE)
print(hw.summary())

## 4. 載入 Pretrained Model
**做什麼**:下載 weights(只在 runtime 下載,存在 Colab 暫存空間,不放進 GitHub)並呼叫 `model.eval()`。**為什麼**:`eval()` 會關閉 Dropout、讓 BatchNorm 使用固定統計量;忘記它會讓結果不穩定。**注意**:第一次執行包含下載時間,這**不屬於** inference latency。

In [ ]:
loaded = lab.load_classifier("mobilenet_v3_small", pretrained=True)
print(f"Model: {loaded.name}, 載入時間 {loaded.load_time_s:.2f}s(含首次下載,不計入 latency)")
print("Input size (C,H,W):", loaded.input_size)

## 5. 載入圖片

In [ ]:
from IPython.display import display
image = lab.load_image(IMAGE_SOURCE)
display(image.resize((256, int(256 * image.height / image.width))))

## 6. Inference 與 Top-5 預測
### 執行流程 (Execution Flow)
```
PIL Image → Resize/Crop/Normalize (CPU) → Tensor [1,3,224,224] → 搬到 Device
          → model(x) forward pass → logits [1,1000] → softmax → Top-K
```
**Technical principle**:`torch.inference_mode()` 關閉 autograd 記錄,節省記憶體與時間。**Preprocessing 必須使用 weights 附帶的 official transforms**,否則輸入分佈與訓練時不同,預測會變差。

In [ ]:
device = lab.select_device(DEVICE)
preds = lab.classify(loaded, image, device, top_k=TOP_K)
for rank, (label, p) in enumerate(preds, 1):
    print(f"{rank}. {label:<30s} {p*100:6.2f}%")
if not loaded.pretrained:
    print("警告:使用隨機 weights,預測無意義。")

## 7. 實驗:分開量測 Preprocessing 與 Model Inference
**做什麼**:用同一張圖,分別計時 preprocessing 與 forward pass。**為什麼**:End-to-end latency 包含兩者,若混在一起會誤判瓶頸在哪裡。這裡只做示範性的單次量測;嚴謹的 benchmark(warm-up、重複、synchronize)在 Lab 02。

In [ ]:
import time, torch
model = loaded.model.to(device).eval()

t0 = time.perf_counter()
x = loaded.transform(image).unsqueeze(0)
t_pre = (time.perf_counter() - t0) * 1000

x = x.to(device)
with torch.inference_mode():
    if device.type == "cuda": torch.cuda.synchronize()   # GPU 是非同步的,計時前後都要 synchronize
    t0 = time.perf_counter()
    model(x)
    if device.type == "cuda": torch.cuda.synchronize()
    t_model = (time.perf_counter() - t0) * 1000
print(f"Preprocessing: {t_pre:.2f} ms | Model forward (單次,含 cold start): {t_model:.2f} ms")

## 8. 預期觀察 (Expected Observations)
- 單次 forward 通常比後續重複執行慢(**cold start**:初始化、cache、GPU context)。你可以重跑上一個 cell 比較。
- Preprocessing 在 CPU 上執行,即使 model 在 GPU,它仍可能佔整體時間的顯著比例。
- 不要用單次量測下結論,請看 Lab 02。

## 9. 常見錯誤 (Common Mistakes)
- 忘記 `model.eval()` 或 `torch.inference_mode()`。
- Model 在 GPU 但 input 在 CPU(**device mismatch**:`Expected all tensors to be on the same device`)。
- GPU 計時沒有 `torch.cuda.synchronize()`,量到的只是「把工作排進佇列」的時間。
- 沒有使用 official preprocessing。

## 10. Troubleshooting
| 問題 | 處理方式 |
|---|---|
| GPU unavailable | 檢查 `Runtime > Change runtime type`;Colab Free 可能暫時無 GPU,改用 CPU 即可 |
| CUDA Out of Memory | 縮小 batch size、`Runtime > Restart session` 釋放記憶體 |
| Package installation failure | 重新執行安裝 cell;確認網路;必要時 `Restart session` 後重跑 |
| Runtime restart | 重啟後從第 2 節重新執行 |
| Model download failure | 重試;若持續失敗,`pretrained=False` 仍可跑速度測試(預測無意義) |
| PyTorch device mismatch | 確認 model 與 input 在同一個 device |
| 量測結果不穩定 | 增加 measure iterations、關閉其他工作、重跑;Colab 是共用環境 |

## 11. 真實產業應用 (Industry Application)
影像分類模型被用於相簿自動分類、製造業瑕疵檢測與內容審核。部署時 inference latency 與 throughput 直接影響成本與使用者體驗。

## 12. 相關概念 (Related Concepts)
Inference vs Training、Preprocessing、Softmax、Cold start、Asynchronous execution(見 `docs/glossary.md`)。

## 官方參考資料 (References)
- [torchvision models](https://pytorch.org/vision/stable/models.html)
- [torch.inference_mode](https://pytorch.org/docs/stable/generated/torch.inference_mode.html)
- [CUDA semantics (asynchronous execution)](https://pytorch.org/docs/stable/notes/cuda.html)